In [1]:
import json
import numpy as np
from tqdm import tqdm


In [2]:
from transformers import AutoProcessor


/home/DTC/conda_pkgs/envs/EHA-mayflower/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/li0007xu/mayflower/Efficient-HA/transformers/src/transformers/utils/hub.py:109: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [9]:

# --- CONFIG ---
POPE_PATH = {
    "random": "../pope_coco/coco_pope_random.json",
    "popular": "../pope_coco/coco_pope_popular.json",
    "adversarial": "../pope_coco/coco_pope_adversarial.json",
}
POPE_TYPE = "adversarial"   # change to "popular" or "adversarial"
MODEL_ID = "google/gemma-3n-E2B-it"   # or your current model
SAMPLE_LIMIT = None  # e.g. 1000 for quick test
# ---------------

processor = AutoProcessor.from_pretrained(MODEL_ID)
pope_path = POPE_PATH[POPE_TYPE]


In [10]:

# Load JSON lines (POPE format)
with open(pope_path, "r") as f:
    data = [json.loads(line) for line in f]

if SAMPLE_LIMIT:
    data = data[:SAMPLE_LIMIT]

lengths = []

print(f"Analyzing {len(data)} samples from {POPE_TYPE} split...")

for entry in tqdm(data, desc="Tokenizing"):
    question = entry["text"]
    image_path = entry["image"]

    # mimic your chat template
    msg = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": question}
        ]
    }]

    text = processor.apply_chat_template(
        msg, tokenize=False, add_generation_prompt=True
    )

    tokens = processor.tokenizer(text, padding=False, truncation=False)
    lengths.append(len(tokens["input_ids"]))


Analyzing 3000 samples from adversarial split...


Tokenizing:   0%|          | 0/3000 [00:00<?, ?it/s]

Tokenizing: 100%|██████████| 3000/3000 [00:00<00:00, 5850.83it/s]


In [11]:

# --- ANALYSIS ---
lengths = np.array(lengths)
print("\n📊 Token length statistics:")
print(f"Samples: {len(lengths)}")
print(f"Min: {lengths.min()}")
print(f"Mean: {lengths.mean():.2f}")
print(f"95th percentile: {np.percentile(lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(lengths, 99):.1f}")
print(f"99.5th percentile: {np.percentile(lengths, 99.5):.1f}")
print(f"Max: {lengths.max()}")

safe_max = int(np.percentile(lengths, 99.5) + 20)
print(f"\n✅ Suggested safe max_length ≈ {safe_max}")


📊 Token length statistics:
Samples: 3000
Min: 19
Mean: 19.20
95th percentile: 20.0
99th percentile: 20.0
99.5th percentile: 20.0
Max: 20

✅ Suggested safe max_length ≈ 40
